# Defining Feature Columns and Target Variables

## Before you train any supervised learning model, you must answer one foundational question with absolute clarity:

### **What exactly am I predicting, and what information am I allowed to use to predict it?**

This question is not philosophical. It is **operational**.

The answer determines:
- ✓ Whether your model is valid
- ✓ Whether your evaluation is honest
- ✓ Whether your system will work in production

**Get this wrong, and nothing that follows — no matter how sophisticated your algorithm — will produce a system that works in the real world.**

## The Discipline

The habit of asking **"would I actually have this information at prediction time?"** is what separates practitioners who build reliable systems from those who build impressive notebooks that fail on deployment.

Many ML failures do not happen because of bad algorithms or insufficient data.  
They happen because:
- The target was defined incorrectly
- Features accidentally included information that wouldn't exist at prediction time
- The separation between features and target was not enforced rigorously

---

# Understanding the Target Variable

The **target variable** is the column your model is trying to predict. It represents the **business outcome of interest**.

In supervised learning, this is the **"correct answer"** provided during training — the ground truth that supervises the learning process.

## The Target is Not Just Another Column

The target is **the reason the ML system exists**. Everything else — features, preprocessing, model selection, evaluation — serves the purpose of predicting this one variable accurately.

### Examples of Target Variables

| Business Problem | Target Column | Type | Values |
|---|---|---|---|
| Customer churn prediction | Churn | Binary classification | Yes, No |
| House price estimation | SalePrice | Regression | 235000, ... |
| Fraud detection | IsFraud | Binary classification | 0 (Legitimate), 1 (Fraud) |
| Delivery time estimation | DeliveryMinutes | Regression | 25, 37, 42, ... |
| Market regime classification | MarketRegime | Multi-class | Bull, Neutral, Bear |

## Three Conditions Every Valid Target Must Satisfy

### ✅ Condition 1: Clearly Defined and Measurable

Vague targets lead to vague models.

- ❌ "Customer satisfaction" — not valid (too vague)
- ✅ "Satisfaction score (1-5)" — valid (specific values)

### ✅ Condition 2: Available in Historical Data

You cannot train supervised model without labeled examples.

- ❌ "Predict customer lifetime value" (if you don't have historical CLV data) — not valid
- ✅ "Predict customer lifetime value" (with historical CLV records) — valid

### ✅ Condition 3: Represents a Real Business Decision or Outcome

The target defines success. If you optimize for the wrong target, you build a system that solves the wrong problem.

- ❌ "Whether customer contacted support" (proxy, but not what matters)
- ✅ "Whether customer will churn" (actual outcome that matters)

---

# Understanding Feature Columns

**Feature columns** are the input variables used to predict the target. These are the **signals the model learns from** — the information available at prediction time that contains predictive patterns about the outcome.

## Examples of Valid Features

In a customer churn dataset:

| Feature | Description | Valid? | Why |
|---|---|---|---|
| Tenure | Number of months as customer | ✅ Yes | Known at prediction time |
| MonthlyCharges | Current monthly bill | ✅ Yes | Known at prediction time |
| ContractType | Month-to-month, annual, etc. | ✅ Yes | Known at prediction time |
| SupportCallsLast90Days | Recent support interactions | ✅ Yes | Known at prediction time |
| CancellationReason | Why customer left | ❌ No | Only exists after churn |
| ChurnDate | Date churn happened | ❌ No | Only exists after churn |

## The Golden Rule of Feature Validity

### 🎯 Features should represent information that is available at prediction time.

This is **not negotiable**. This is the **foundational principle** that separates valid ML systems from broken ones.

### The Test: Imagine the Real-World Prediction Moment

**Close your eyes and imagine the moment in the real world when you need to make a prediction.**

What information do you have at that exact moment?
- **Those are your valid features.**

Everything else is **leakage**.

### Example: Predicting Loan Default

**Prediction moment**: Loan application is submitted. Decision needed: Approve or Reject?

**Valid features at that moment:**
- CreditScore (applicant's official credit score)
- Income (reported annual income)
- DebtToIncomeRatio (debt relative to income)
- EmploymentLength (years at current job)
- LoanAmount (requested loan amount)

**Invalid features (leakage):**
- ❌ DaysUntilDefault (you don't know if/when default occurs until after)
- ❌ DefaultAmount (only known if default happens)
- ❌ RepaymentHistory (doesn't exist until after loan is issued)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("=" * 80)
print("BASIC FEATURE AND TARGET SEPARATION")
print("=" * 80)

# Create example dataset: Customer Churn
np.random.seed(42)
n_customers = 1000

df = pd.DataFrame({
    'CustomerID': range(1, n_customers + 1),
    'Tenure': np.random.randint(1, 72, n_customers),
    'MonthlyCharges': np.random.uniform(20, 120, n_customers),
    'TotalCharges': np.random.uniform(100, 8000, n_customers),
    'ContractType': np.random.choice(['Month-to-month', 'One year', 'Two year'], n_customers),
    'SupportCallsLast90Days': np.random.randint(0, 10, n_customers),
    'Churn': np.random.choice(['Yes', 'No'], n_customers),  # TARGET
    'CancellationReason': ['Leakage: Only exists after churn'] * n_customers  # DO NOT USE
})

print("\nDataset shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

print("\n" + "=" * 80)
print("APPROACH 1: Basic Separation")
print("=" * 80)

# Define target explicitly
TARGET_COLUMN = "Churn"

# WRONG approach (not recommended)
# X_wrong = df.drop(columns=[TARGET_COLUMN])  # Still includes CustomerID and CancellationReason!

# RIGHT approach (explicit features)
NUMERICAL_FEATURES = ['Tenure', 'MonthlyCharges', 'TotalCharges', 'SupportCallsLast90Days']
CATEGORICAL_FEATURES = ['ContractType']
EXCLUDED_COLUMNS = ['CustomerID', 'CancellationReason']

ALL_FEATURES = NUMERICAL_FEATURES + CATEGORICAL_FEATURES

print(f"\nTarget: {TARGET_COLUMN}")
print(f"Numerical features: {NUMERICAL_FEATURES}")
print(f"Categorical features: {CATEGORICAL_FEATURES}")
print(f"Excluded columns: {EXCLUDED_COLUMNS}")

# Separate X and y
X = df[ALL_FEATURES]
y = df[TARGET_COLUMN]

print(f"\n✓ Separation complete:")
print(f"  - Features (X): {X.shape}")
print(f"  - Target (y): {y.shape}")
print(f"  - Target values: {y.unique()}")
print(f"  - Target distribution:\n{y.value_counts(normalize=True)}")

# Validate separation
print("\n" + "=" * 80)
print("VALIDATION")
print("=" * 80)

# Check 1: Target not in features
assert TARGET_COLUMN not in X.columns, "❌ ERROR: Target leaked into features!"
print(f"✓ Target NOT in features")

# Check 2: Excluded columns not in features
for col in EXCLUDED_COLUMNS:
    assert col not in X.columns, f"❌ ERROR: Excluded column '{col}' in features!"
print(f"✓ Excluded columns NOT in features")

# Check 3: All features exist in data
missing = set(ALL_FEATURES) - set(df.columns)
assert not missing, f"❌ ERROR: Features not found: {missing}"
print(f"✓ All features found in dataset")

# Check 4: No NaN in target
assert y.notna().all(), "❌ ERROR: Target contains NaN values!"
print(f"✓ Target has no missing values")

---

# Feature Eligibility Criteria

Not every column in a dataset should automatically become a feature. Evaluate each carefully.

## Valid Features Must Be:

✅ **Available at prediction time** — you can compute or access this value when predicting  
✅ **Relevant to the target** — contains information that could help predict the outcome  
✅ **Not directly encoding the target** — not derived from the target  
✅ **Not a post-outcome variable** — doesn't represent information only existing after outcome  
✅ **Good data quality** — not 99% missing or entirely duplicated

## Columns to Treat Cautiously

### ❌ Category 1: IDs and Unique Identifiers

**Examples**: CustomerID, TransactionID, OrderID, SessionID

**Problem**: These uniquely identify each row but carry no generalizable signal. A model that memorizes customer IDs learns nothing useful for new customers.

**Decision**: Exclude (unless ID encodes meaningful information like ProductCategory from ProductID)

### ❌ Category 2: Raw Timestamps

**Examples**: OrderDate, TransactionTimestamp, SignupDate

**Problem**: The model cannot learn from literal timestamps (2024-03-15 14:23:17) in a generalizable way.

**Solution**: Derive meaningful features instead:
- `DayOfWeek` — Monday = 1, Tuesday = 2, etc.
- `HourOfDay` — 0-23
- `IsWeekend` — binary flag
- `MonthOfYear` — 1-12
- `DaysSinceSignup` — tenure measure
- `DaysSinceLastPurchase` — recency measure

**Decision**: Exclude raw timestamps; include derived temporal features

### ❌ Category 3: Free-Text Descriptions

**Examples**: CustomerComments, ProductDescription, EmailBody

**Problem**: Text cannot be used directly; models require numerical inputs.

**Solution**: Transform text into numerical features:
- TF-IDF vectors
- Word embeddings (Word2Vec, GloVe)
- Sentiment scores (-1 to +1)
- Text length
- Presence of keywords

**Decision**: Exclude raw text; include text-derived numerical features

### ❌ Category 4: Target-Derived Columns (LEAKAGE)

**Examples**:
- `DaysUntilChurn` in churn prediction (calculated from churn date)
- `DefaultAmount` in loan default (only known if default occurs)
- `FraudScore` in fraud detection (if computed from fraud labels)

**Problem**: These are calculated FROM or only exist BECAUSE the target occurred. Perfect predictor in training, completely unavailable in production.

**Detection**: If feature correlation with target > 0.95, investigate for leakage. If feature wouldn't exist before outcome, it's leakage.

**Decision**: Always exclude target-derived columns

### ❌ Category 5: Post-Event Data

**Example**: Predicting hospital readmission and including "FollowUpAppointmentBooked" if scheduled based on predicted readmission risk.

**Problem**: Circular dependency. Feature generated based on prediction you're trying to make.

**Decision**: Exclude any information generated after prediction decision point

---

# Common Mistakes in Feature and Target Definition

## ❌ Mistake 1: Including the Target as a Feature

In [ ]:
print("=" * 80)
print("COMMON MISTAKES & HOW TO AVOID THEM")
print("=" * 80)

# ❌ MISTAKE 1: Including Target as Feature
print("\n" + "=" * 80)
print("MISTAKE 1: Including Target as Feature")
print("=" * 80)

print("\n❌ WRONG: Only drops ID, keeps target in features")
X_wrong = df.drop(columns=['CustomerID'])
print(f"Shape: {X_wrong.shape}")
print(f"Columns: {X_wrong.columns.tolist()}")
print("⚠️  Churn is in the features! Model will have 100% training accuracy but fail in production")

print("\n✅ RIGHT: Drop both ID and target")
X_correct = df.drop(columns=['CustomerID', 'Churn', 'CancellationReason'])
y_correct = df['Churn']
print(f"Features shape: {X_correct.shape}")
print(f"Target shape: {y_correct.shape}")
print(f"Churn in features? {('Churn' in X_correct.columns)}")

# ❌ MISTAKE 2: Including Post-Outcome Columns
print("\n" + "=" * 80)
print("MISTAKE 2: Including Post-Outcome Columns (Leakage)")
print("=" * 80)

print("\n❌ WRONG: Including CancellationReason column")
print("This column only EXISTS because churn happened (post-outcome)")
print("Model learns by reading CancellationReason, which won't exist at prediction time")
df_wrong = df[['Tenure', 'MonthlyCharges', 'CancellationReason', 'Churn']]
print(f"Features: {df_wrong.columns.tolist()}")

print("\n✅ RIGHT: Use only pre-outcome information")
df_right = df[['Tenure', 'MonthlyCharges', 'ContractType', 'Churn']]
print(f"Features: {df_right.columns.tolist()}")

# ❌ MISTAKE 3: Not Removing Identifiers
print("\n" + "=" * 80)
print("MISTAKE 3: Not Removing Identifiers")
print("=" * 80)

print("\n❌ WRONG: Including CustomerID")
print("Model memorizes specific customer IDs from training")
print("Useless for predicting on new (unseen) customers")

# Show correlation
print("\nCustomerID correlation with target (meaningless):")
print(f"  - Calculated statistic: {df['CustomerID'].corr(df['Churn'].astype(int)):.4f}")
print("  - This is not a real signal; it's overfitting to the training set")

print("\n✅ RIGHT: Exclude CustomerID")
print("Focus on customer characteristics (Tenure, Charges, Contract, etc.)")
print("These generalize to new customers")

# ❌ MISTAKE 4: Raw Timestamps Without Derivation
print("\n" + "=" * 80)
print("MISTAKE 4: Raw Timestamps Without Derivation")
print("=" * 80)

# Create timestamp example
df_ts = df.copy()
df_ts['SignupDate'] = pd.date_range('2020-01-01', periods=len(df), freq='H')

print("\n❌ WRONG: Using raw timestamp directly")
print(f"Raw timestamp example: {df_ts['SignupDate'].iloc[0]}")
print("Model cannot learn generalizable patterns from specific dates")

print("\n✅ RIGHT: Derive meaningful temporal features")
df_ts['DaysSinceSignup'] = (pd.Timestamp.now() - df_ts['SignupDate']).dt.days
df_ts['SignupMonth'] = df_ts['SignupDate'].dt.month
df_ts['SignupDayOfWeek'] = df_ts['SignupDate'].dt.dayofweek

print(f"Derived features:")
print(f"  - DaysSinceSignup: {df_ts['DaysSinceSignup'].iloc[0]} (tenure in days)")
print(f"  - SignupMonth: {df_ts['SignupMonth'].iloc[0]} (seasonal pattern)")
print(f"  - SignupDayOfWeek: {df_ts['SignupDayOfWeek'].iloc[0]} (day of week)")
print("These features capture temporal patterns that generalize")

---

# Data Leakage: The Silent Killer

Data leakage is the most dangerous mistake in supervised learning.

## Leakage Produces Models That:

- ✗ Achieve 95%+ accuracy in training
- ✗ Achieve 95%+ accuracy in cross-validation
- ✗ Achieve 50% accuracy (or worse) in production

**You never discover the problem until deployment.**

## The Correct Sequence to Prevent Leakage

```
1. Load data
2. Define target and features explicitly ← YOU ARE HERE
3. Remove excluded columns
4. Separate X and y
5. Split into train/test (BEFORE any fitting)
6. Fit preprocessing on training data ONLY
7. Apply preprocessing to test data
8. Train model on preprocessed training data
9. Evaluate on preprocessed test data
```

## Critical: Split BEFORE Fitting Preprocessing

### ❌ WRONG: Fit Preprocessing on All Data

```python
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # Fit on ALL data (includes test set!)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y)
```

**Problem**: Scaler computed mean & std using test set statistics. Information from test set leaked into training.

### ✅ RIGHT: Split First, Fit on Train Only

```python
X_train, X_test, y_train, y_test = train_test_split(X, y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # Fit ONLY on train
X_test_scaled = scaler.transform(X_test)       # Apply to test
```

**Result**: Test set remains truly unseen. No leakage.

In [ ]:
print("\n" + "=" * 80)
print("DEMONSTRATING CORRECT TRAIN/TEST SEPARATION (NO LEAKAGE)")
print("=" * 80)

# Step 1: Separate features and target
print("\nStep 1: Separate X and y")
X = df[['Tenure', 'MonthlyCharges', 'TotalCharges', 'SupportCallsLast90Days']]
y = df['Churn']
print(f"X shape: {X.shape}, y shape: {y.shape}")

# Step 2: Split BEFORE preprocessing
print("\nStep 2: Split train/test BEFORE preprocessing")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

# Step 3: Fit preprocessing ONLY on training data
print("\nStep 3: Fit scaler on training data only")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
print(f"Scaler fitted on training data")
print(f"Scaler mean (learned from train): {scaler.mean_}")
print(f"Scaler std (learned from train): {scaler.scale_}")

# Step 4: Apply same preprocessing to test data
print("\nStep 4: Transform test data using fitted scaler")
X_test_scaled = scaler.transform(X_test)
print(f"Test data transformed (no leakage)")

# Verify no leakage
print("\n" + "=" * 80)
print("VERIFICATION: No Leakage Occurred")
print("=" * 80)

# Check 1: Test set statistics should be DIFFERENT from scaler statistics
train_mean = X_train[[0]].mean().values[0]
test_mean = X_test[[0]].mean().values[0]
scaler_mean = scaler.mean_[0]

print(f"\nFeature 0 (Tenure) statistics:")
print(f"  - Training set mean: {train_mean:.2f}")
print(f"  - Test set mean: {test_mean:.2f}")
print(f"  - Scaler was fitted on: {scaler_mean:.2f} (training mean)")
print(f"  - ✓ Scaler does NOT know about test set statistics")

# Check 2: Original train/test sets remain unchanged
assert len(X_train) + len(X_test) == len(X), "Train/test don't partition original data"
print(f"\n✓ Train and test sets properly partition original data")

# Check 3: No overlap between train and test
assert len(set(X_train.index) & set(X_test.index)) == 0, "Train/test overlap detected!"
print(f"✓ No overlap between train and test sets")

print("\n✓✓✓ CORRECT SEPARATION: No data leakage")

---

# Categorical vs Numerical vs Ordinal Features

After identifying valid feature columns, categorize them by data type. This determines preprocessing.

## Numerical Features

**Continuous values**: Age, Salary, MonthlyCharges, Temperature  
**Integer counts**: NumberOfPurchases, YearsExperience, Tenure

- Can be used directly by most models
- May require scaling or normalization
- Can be binned into categories if needed

## Categorical Features

**Discrete labels with NO intrinsic ordering**: Gender, ContractType, PaymentMethod, Region, ProductCategory

- Must be encoded (one-hot, label encoding, target encoding) before use
- No natural ordering
- Cannot be used directly with distance-based models

## Ordinal Features (Special Case)

**Discrete labels WITH natural ordering**: EducationLevel (High School < Bachelor < Master < PhD), SatisfactionRating (Very Dissatisfied < ... < Very Satisfied)

- Can be label-encoded with meaningful integer mappings
- Preserve ordering in encoding
- More specialized handling than categorical

## Explicit Configuration Example

```python
# config.py
TARGET_COLUMN = 'Churn'

NUMERICAL_FEATURES = [
    'tenure',
    'MonthlyCharges',
    'TotalCharges'
]

CATEGORICAL_FEATURES = [
    'gender',
    'Contract',
    'PaymentMethod',
    'InternetService',
]

ORDINAL_FEATURES = {
    'SatisfactionRating': ['Very Dissatisfied', 'Dissatisfied', 'Neutral', 'Satisfied']
}

EXCLUDED_COLUMNS = ['CustomerID']

ALL_FEATURES = NUMERICAL_FEATURES + CATEGORICAL_FEATURES + list(ORDINAL_FEATURES.keys())
```

### Benefits of Explicit Configuration:

✅ Prevents silent mistakes (encoding numerical as categorical)  
✅ Makes preprocessing reproducible  
✅ Makes definitions reviewable  
✅ Makes code modular (feature lists imported, not scattered)

In [ ]:
print("\n" + "=" * 80)
print("FEATURE VALIDATION FUNCTION")
print("=" * 80)

def validate_features(df, target_col, feature_cols):
    """
    Validate feature and target definitions.
    
    Checks for:
    - Target not in features
    - No ID-like columns in features
    - No suspiciously high correlations (possible leakage)
    - All features exist in dataset
    - Target exists in dataset
    """
    
    print(f"\nValidating feature definitions...")
    print(f"Dataset shape: {df.shape}")
    
    # Check 1: Target in dataset
    if target_col not in df.columns:
        raise ValueError(f"❌ Target '{target_col}' not found in dataset!")
    print(f"✓ Target '{target_col}' found in dataset")
    
    # Check 2: Target not in features
    if target_col in feature_cols:
        raise ValueError(f"❌ Target '{target_col}' found in feature list!")
    print(f"✓ Target NOT in features")
    
    # Check 3: Check for ID patterns
    id_patterns = ['id', 'ID', 'Id', 'key', 'KEY', 'Key', 'identifier']
    potential_ids = [col for col in feature_cols 
                     if any(pattern in col for pattern in id_patterns)]
    if potential_ids:
        print(f"⚠️  Warning: Potential ID columns found: {potential_ids}")
        print(f"   These should likely be excluded")
    
    # Check 4: Check for high correlation with target (possible leakage)
    numeric_features = [col for col in feature_cols 
                        if col in df.select_dtypes(include=['number']).columns]
    
    if numeric_features and target_col in df.select_dtypes(include=['number']).columns:
        high_corr_features = []
        for col in numeric_features:
            corr = df[col].corr(df[target_col])
            if abs(corr) > 0.95:
                high_corr_features.append((col, corr))
        
        if high_corr_features:
            print(f"⚠️  WARNING: Features with high target correlation (>0.95):")
            for col, corr in high_corr_features:
                print(f"   - {col}: {corr:.4f} *** POSSIBLE LEAKAGE ***")
    
    # Check 5: All features exist in dataset
    missing = set(feature_cols) - set(df.columns)
    if missing:
        raise ValueError(f"❌ Features not found in dataset: {missing}")
    print(f"✓ All {len(feature_cols)} features found in dataset")
    
    # Check 6: No missing values in target
    missing_target = df[target_col].isnull().sum()
    if missing_target > 0:
        print(f"⚠️  Warning: {missing_target} missing values in target ({missing_target/len(df)*100:.1f}%)")
    else:
        print(f"✓ Target has no missing values")
    
    print(f"\n✓✓✓ Feature validation passed")
    print(f"    - {len(feature_cols)} features validated")
    print(f"    - Target: {target_col}")

# Run validation
validate_features(df, 'Churn', ['Tenure', 'MonthlyCharges', 'TotalCharges', 'SupportCallsLast90Days', 'ContractType'])

---

# Applying This Framework to StockAxis

## The Critical Question for StockAxis

**What exactly am I predicting, and what information am I allowed to use?**

### Your Target Variable

Before proceeding, answer clearly:

- **Target column name**: [What column are you predicting?]
- **Prediction goal**: [Predict what outcome?]
- **Values**: [What are the possible values?]
- **Business meaning**: [What does each value represent?]

**Example possibilities for stock market:**
- `PriceDirection`: Binary classification (UP/DOWN)
- `MarketRegime`: Multi-class classification (BULL/NEUTRAL/BEAR)
- `Return_Next5Days`: Regression (continuous return percentage)

### Your Feature Columns (Prediction Time Question)

For each potential feature in your dataset, ask:

**"At the exact moment I need to predict, would I have this information?"**

Valid features might include:
- ✅ Historical price data (available at prediction time)
- ✅ Technical indicators computed on past data (RSI, MACD, etc.)
- ✅ Volume patterns (historical, not future)
- ✅ Time-based features (day of week, time of day)
- ✅ Relative performance vs benchmark (calculated from available data)

Invalid features would include:
- ❌ Future price movements (would happen after prediction)
- ❌ Actual outcome (would be circular)
- ❌ Indicators computed with lookahead bias
- ❌ Information that only exists after the trading decision

### Temporal Leakage in Stock Trading

Stock market predictions are particularly prone to **temporal leakage** because:

❌ **WRONG**: Using `ClosePrice_T+1` as a feature to predict `PriceDirection_T`
- You're predicting tomorrow's direction using tomorrow's close price
- Completely invalid; you can't know future prices

✅ **RIGHT**: Using `ClosePrice_T`, `Volume_T`, `RSI_T-1` to predict `PriceDirection_T+1`
- Using information available at time T to predict time T+1
- This is valid temporal separation

In [ ]:
print("\n" + "=" * 80)
print("STOCKAXIS: EXAMPLE FEATURE DEFINITION")
print("=" * 80)

# Example: Stock price prediction dataset
np.random.seed(42)
n_days = 500

stockaxis_df = pd.DataFrame({
    'Date': pd.date_range('2023-01-01', periods=n_days, freq='D'),
    'Ticker': ['AAPL'] * n_days,
    'OpenPrice': np.random.uniform(150, 180, n_days),
    'ClosePrice': np.random.uniform(150, 180, n_days),
    'Volume': np.random.uniform(1e6, 5e6, n_days),
    'RSI_14': np.random.uniform(20, 80, n_days),  # Relative Strength Index
    'MACD': np.random.uniform(-5, 5, n_days),     # MACD indicator
    'DayOfWeek': np.random.randint(0, 5, n_days),
    'VIX': np.random.uniform(10, 30, n_days),     # Volatility index
    # These are INVALID (future information / leakage)
    'ClosePrice_Tomorrow': np.random.uniform(150, 180, n_days),
    'PriceDirection_Tomorrow': np.random.choice([0, 1], n_days),  # TARGET or LEAKAGE?
})

print("\nStockAxis Dataset (first 5 rows):")
print(stockaxis_df.head())

print("\n" + "=" * 80)
print("FEATURE CATEGORIZATION FOR STOCKAXIS")
print("=" * 80)

STOCKAXIS_TARGET = 'PriceDirection_Tomorrow'

STOCKAXIS_NUMERICAL_FEATURES = [
    'OpenPrice',           # Current day's opening price
    'ClosePrice',          # Current day's closing price (known at market close)
    'Volume',              # Current day's volume (known at market close)
    'RSI_14',              # Technical indicator (historical calculation)
    'MACD',                # Technical indicator (historical calculation)
    'VIX',                 # Current volatility (available at market close)
]

STOCKAXIS_CATEGORICAL_FEATURES = [
    'DayOfWeek',           # 0=Monday through 4=Friday
]

STOCKAXIS_EXCLUDED_COLUMNS = [
    'Ticker',                      # Identifier (single stock)
    'Date',                        # Raw timestamp (derive features instead)
    'ClosePrice_Tomorrow',         # FUTURE DATA - LEAKAGE!
    # Note: PriceDirection_Tomorrow is TARGET, not excluded
]

STOCKAXIS_ALL_FEATURES = STOCKAXIS_NUMERICAL_FEATURES + STOCKAXIS_CATEGORICAL_FEATURES

print(f"\n📊 StockAxis Configuration:")
print(f"\nTARGET: {STOCKAXIS_TARGET}")
print(f"  Type: Binary Classification (0=Down, 1=Up)")
print(f"  Available: At end of current trading day")

print(f"\nNUMERICAL FEATURES ({len(STOCKAXIS_NUMERICAL_FEATURES)}):")
for feat in STOCKAXIS_NUMERICAL_FEATURES:
    print(f"  ✓ {feat}")

print(f"\nCATEGORICAL FEATURES ({len(STOCKAXIS_CATEGORICAL_FEATURES)}):")
for feat in STOCKAXIS_CATEGORICAL_FEATURES:
    print(f"  ✓ {feat}")

print(f"\nEXCLUDED COLUMNS ({len(STOCKAXIS_EXCLUDED_COLUMNS)}):")
for col in STOCKAXIS_EXCLUDED_COLUMNS:
    print(f"  ✗ {col}")

# Validate
print("\n" + "=" * 80)
print("VALIDATION")
print("=" * 80)

assert STOCKAXIS_TARGET not in STOCKAXIS_ALL_FEATURES, "Target in features!"
assert set(STOCKAXIS_EXCLUDED_COLUMNS).isdisjoint(set(STOCKAXIS_ALL_FEATURES)), "Excluded in features!"
assert set(STOCKAXIS_ALL_FEATURES).isdisjoint(set(STOCKAXIS_EXCLUDED_COLUMNS)), "Features/excluded overlap!"

print(f"✓ Target not in features")
print(f"✓ Excluded columns not in features")
print(f"✓ No feature/excluded overlap")
print(f"✓ Total features: {len(STOCKAXIS_ALL_FEATURES)}")

# Separate X and y
X_stockaxis = stockaxis_df[STOCKAXIS_ALL_FEATURES]
y_stockaxis = stockaxis_df[STOCKAXIS_TARGET]

print(f"\n✓ Separation complete:")
print(f"  - Features (X): {X_stockaxis.shape}")
print(f"  - Target (y): {y_stockaxis.shape}")
print(f"  - Target distribution: {y_stockaxis.value_counts().to_dict()}")

---

# Best Practices Summary

## Practice 1: Centralize Configuration

Define all features, target, and excluded columns in a central config file:

```python
# config.py
TARGET_COLUMN = 'YourTarget'
NUMERICAL_FEATURES = [...]
CATEGORICAL_FEATURES = [...]
EXCLUDED_COLUMNS = [...]
ALL_FEATURES = NUMERICAL_FEATURES + CATEGORICAL_FEATURES
```

Every module imports from this single source of truth.

## Practice 2: Validate Feature Definitions

Write validation functions that check for common errors **before** training.

## Practice 3: Document Feature Availability

For each feature, document when it becomes available in your business process.

## Practice 4: Simulate the Prediction Scenario

Mentally simulate the real-world prediction moment. What information exists then?

## Practice 5: Enforce Separation Rigorously

```python
# ALWAYS separate before preprocessing
X_train, X_test, y_train, y_test = train_test_split(X, y)

# ALWAYS fit preprocessing on train only
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
```

No exceptions. Ever.

---

# Key Takeaways

## Before You Train Any Model:

✓ Clearly identify the target variable  
✓ Define what each target value means in business terms  
✓ Identify all legitimate features (available at prediction time)  
✓ Remove all leakage (target-derived, post-outcome, future information)  
✓ Remove metadata (IDs, raw timestamps)  
✓ Derive meaningful features (temporal, text, etc.)  
✓ Separate X and y explicitly  
✓ Validate no target leakage  
✓ Document everything  

## The Non-Negotiable Sequence:

1. Define target and features
2. Separate X and y
3. **Split into train/test**
4. Fit preprocessing on train only
5. Apply preprocessing to test
6. Train model
7. Evaluate on test

**Get this right. Everything else depends on it.**

---

# Practical Exercise: Your Project

### Part 1: Answer These Questions

1. **What is your target variable?**
   - Column name?
   - What does it represent?
   - Classification or regression?
   - [Your answer]

2. **Which columns are valid features?**
   - Why is each available at prediction time?
   - [Your answer]

3. **Which columns are excluded and why?**
   - IDs? Leakage? Post-outcome? Raw timestamps?
   - [Your answer]

4. **Could you have data leakage?**
   - Any features derived from target?
   - Any future information?
   - Any post-outcome data?
   - [Your answer]

### Part 2: Implement While Reviewing Your Data

```python
import pandas as pd
from sklearn.model_selection import train_test_split

# Load your data
df = pd.read_csv('your_data.csv')

# Answer the questions above
TARGET_COLUMN = 'your_target'
NUMERICAL_FEATURES = ['feature1', 'feature2', ...]
CATEGORICAL_FEATURES = ['feature3', 'feature4', ...]
EXCLUDED_COLUMNS = ['id', 'reason_for_exclusion', ...]
ALL_FEATURES = NUMERICAL_FEATURES + CATEGORICAL_FEATURES

# Validate
assert TARGET_COLUMN not in ALL_FEATURES
assert set(EXCLUDED_COLUMNS).isdisjoint(set(ALL_FEATURES))

# Separate
X = df[ALL_FEATURES]
y = df[TARGET_COLUMN]

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

print(f"✓ Features: {X.shape}")
print(f"✓ Target: {y.shape}")
```

This is your foundation. Everything that follows depends on getting this right.